# Zambia Inflation and Cost of Living (2020–2026)
## Exploratory analysis of monthly consumer price data

**Author:** Canan Phiri  
**Period:** January 2020–October 2026  
**Source:** Zambia Statistics Agency (ZamStats), *The Monthly*, Vol. 256 (July 2024) and Vol. 283 (October 2026).

This notebook provides a reproducible descriptive analysis of the CPI dataset in `data/processed/zambia_cpi_monthly_2020_2026.csv`.

### Questions
1. How did annual headline inflation evolve?
2. How did food and non-food price indices change relative to January 2020?
3. How large was the accumulated change in the overall CPI?
4. What can CPI data tell us—and what can it not tell us—about household purchasing power?

**Interpretation note:** A decline in annual inflation means prices are increasing more slowly; it does not, by itself, mean that the overall price level has fallen. CPI data alone cannot establish changes in individual household purchasing power because that also depends on income and expenditure patterns.


## 1. Set-up and libraries

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Locate the repository root whether this notebook is run from the root or analysis/.
candidates = [Path.cwd(), Path.cwd().parent]
repo_root = next((p for p in candidates if (p / "data/processed/zambia_cpi_monthly_2020_2026.csv").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from the repository root or analysis/ folder.")

data_path = repo_root / "data/processed/zambia_cpi_monthly_2020_2026.csv"
figures_dir = repo_root / "figures"
figures_dir.mkdir(exist_ok=True)
print("Dataset:", data_path)


## 2. Load and inspect the data

In [ ]:
df = pd.read_csv(data_path)
df["month"] = pd.to_datetime(df["month"], format="%Y-%m")
df = df.sort_values("month").reset_index(drop=True)

print("Observations:", len(df))
print("Period:", df["month"].min().strftime("%B %Y"), "to", df["month"].max().strftime("%B %Y"))
display(df.head())
display(df.tail())


### Data quality checks

In [ ]:
print("Duplicate month records:", df["month"].duplicated().sum())
print("\\nMissing values by column:")
display(df.isna().sum().to_frame("missing_count"))

expected_months = pd.date_range(df["month"].min(), df["month"].max(), freq="MS")
missing_months = expected_months.difference(df["month"])
print("Missing calendar months:", list(missing_months.strftime("%Y-%m")))
display(df[["source_document", "verification_status"]].drop_duplicates())


## 3. Descriptive statistics

In [ ]:
key_cols = [
    "headline_inflation_yoy_pct", "food_inflation_yoy_pct", "nonfood_inflation_yoy_pct",
    "cpi_all_jan2020_100", "cpi_food_jan2020_100", "cpi_nonfood_jan2020_100"
]
display(df[key_cols].describe().T)


## 4. Annual headline inflation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(df["month"], df["headline_inflation_yoy_pct"], linewidth=2)
ax.set_title("Zambia: Annual Headline Inflation, 2020–2026")
ax.set_xlabel("Month")
ax.set_ylabel("Annual inflation (%)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(figures_dir / "01_headline_inflation_trend.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Food and non-food price indices

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(df["month"], df["cpi_food_jan2020_100"], label="Food CPI", linewidth=2)
ax.plot(df["month"], df["cpi_nonfood_jan2020_100"], label="Non-food CPI", linewidth=2)
ax.set_title("Zambia: Food and Non-food Price Indices (January 2020 = 100)")
ax.set_xlabel("Month")
ax.set_ylabel("Index (January 2020 = 100)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(figures_dir / "02_food_vs_nonfood_prices.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Accumulated change in the overall price level

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(df["month"], df["cpi_all_jan2020_100"], linewidth=2)
ax.axhline(100, linestyle="--", linewidth=1)
ax.set_title("Zambia: Accumulated Change in Consumer Prices (January 2020 = 100)")
ax.set_xlabel("Month")
ax.set_ylabel("Overall CPI (January 2020 = 100)")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(figures_dir / "03_accumulated_consumer_prices.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Calculate cumulative price changes

In [ ]:
first, last = df.iloc[0], df.iloc[-1]
summary = pd.DataFrame({
    "CPI component": ["Overall", "Food", "Non-food"],
    "January 2020 index": [first["cpi_all_jan2020_100"], first["cpi_food_jan2020_100"], first["cpi_nonfood_jan2020_100"]],
    "Latest index": [last["cpi_all_jan2020_100"], last["cpi_food_jan2020_100"], last["cpi_nonfood_jan2020_100"]],
})
summary["Cumulative change (%)"] = (summary["Latest index"] / summary["January 2020 index"] - 1) * 100
display(summary.round(2))


## 8. Selected inflation observations

In [ ]:
selected = df[
    df["month"].dt.month.isin([1, 12]) | (df["month"] == df["month"].max())
][["month", "headline_inflation_yoy_pct", "food_inflation_yoy_pct", "nonfood_inflation_yoy_pct"]].copy()
selected["month"] = selected["month"].dt.strftime("%b %Y")
display(selected.reset_index(drop=True))


## 9. Interpretation and limitations

### What the CPI series can show
- The direction and pace of annual consumer price inflation.
- The relative movement of food and non-food CPI components.
- The accumulated change in CPI relative to January 2020.

### What it cannot establish on its own
- Whether every household experienced the same cost-of-living change.
- Whether wages or disposable incomes kept pace with consumer prices.
- The causal contribution of exchange rates, monetary policy, fuel prices or other factors.
- A formal exchange-rate pass-through relationship.

A household purchasing-power assessment requires income data and a clearly specified household expenditure basket. Causal claims require an appropriate econometric design and additional variables.

## 10. Reproducibility and sources

The input data are stored in `data/processed/`. Source publications and documentation are listed in `references/sources.md`. The CPI series covers January 2020–October 2026 and contains 82 monthly observations.

Derived rates and rebased indices should be checked against the source publications before the results are used in formal policy or academic publication.
